# FLUX.2 Klein 4B – Master Clinical Viewpoint Generator
One notebook: generates the **canonical frontal image**, then edits that same frontal image into **Left 20/40/60, Right 20/40/60, Basal and Helicopter** views.

### What was wrong in the two original notebooks (and what this notebook fixes)
1. **Prompts were silently cut off.** FLUX.2 Klein encodes the prompt with a hard limit of **512 tokens** (including the chat template) and truncates the end without any warning. Measured with the Qwen tokenizer: the old basal/helicopter prompts in the end-to-end notebook were 747–877 tokens, the old Left 60° prompt 892 tokens (43 % never seen). The "alt1/alt2" retry text and the "correction" paragraphs sat at the very end, so they were never read. **All prompts here are 470–490 tokens or less, with the critical viewpoint instructions first, and the notebook checks the budget before using the GPU.**
2. **Left/right direction was contradictory.** The notebooks mixed *anatomical* wording ("patient's own left", "viewer's left") with *image* wording ("nose toward image-left"), and Left 20/40 and Left 60 pointed the nose in opposite directions. Prompts that mixed both turned the head the wrong way. **Now every prompt uses image-edge wording only** (one convention, below).
3. **Basal view failed in the end-to-end notebook.** It was sent behind a generic prefix that says "change only head orientation… no camera tilt", which contradicts a camera-below view, plus truncation. **Basal and helicopter now have their own prompts without that prefix.**
4. **Broken/undefined code.** The saved prompt cell of the end-to-end notebook had a syntax error (half-converted dict) and the testing notebook used an undefined `reference_image`. **Fixed.**
5. **Numeric angles are only a request.** The tests show the model does not obey degrees exactly (the "55° for 40°" calibration was not supported: 40° and 55° with the same seed gave nearly the same turn; 95° for 60° on the left side did give a ~60° look). Calibrated values live in `PROMPT_DEGREES`.

### Direction convention (single, used everywhere)
| View | Nose points toward | Side facing the camera |
|---|---|---|
| Left 20/40/60 | **LEFT edge of the image** | patient's left cheek/ear (right side of image) |
| Right 20/40/60 | **RIGHT edge of the image** | patient's right cheek/ear (left side of image) |

Prompt-based editing cannot guarantee exact angles or identity: **always review the images manually.** Not a clinical tool.

In [ ]:
# ============================================================
# CENTRAL CONFIGURATION  (change only DEFAULT_BATCH_ID / DEFAULT_SEED normally)
# ============================================================
import os
from pathlib import Path

DEFAULT_BATCH_ID = "P05_batch"     # P05_batch -> patient P05
DEFAULT_SEED = 20261034
USE_BACKEND_MANIFEST = True       # True: /kaggle/input/**/manifest.json controls the frontal prompt/seed

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"
_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
BASE = _ROOT / "rhinoplasty_dataset"
MANIFEST_SEARCH_ROOT = Path("/kaggle/input")

# Frontal (canonical reference) settings - as in the end-to-end notebook
FRONTAL_WIDTH, FRONTAL_HEIGHT = 384, 384
FRONTAL_STEPS, FRONTAL_GUIDANCE = 12, 1.0

# Viewpoint settings. 8 steps = the setting used for the approved basal/helicopter tests.
VIEW_WIDTH, VIEW_HEIGHT = 512, 512
VIEW_STEPS, VIEW_GUIDANCE = 8, 1.0
MAX_PROMPT_ATTEMPTS = 3            # technical retries (new seed each time)
SKIP_EXISTING = True
FORCE_REGENERATE_VIEWS = set()     # e.g. {"basal","helicopter"}

# Number written into each prompt (a REQUEST, not a guarantee of the real angle).
# left_60 = 95 was the only value that visually gave ~60 deg in the tests; 55 for 40 was not supported.
# Right-side 60 over-rotated to near-profile in the test (see notes) - lower it if you see profile views.
PROMPT_DEGREES = {"left_20": 20, "left_40": 40, "left_60": 95,
                  "right_20": 20, "right_40": 40, "right_60": 60}

# Seeds (same scheme as the end-to-end notebook)
VIEW_SEED_OFFSET, VIEW_SEED_STRIDE, ATTEMPT_SEED_STRIDE, RERUN_SEED_OFFSET = 1000, 100, 17, 9999

MAX_PROMPT_TOKENS = 500            # hard model limit is 512 incl. ~12 template tokens
INSTALL_DEPENDENCIES = True
print("Batch:", DEFAULT_BATCH_ID, "| seed:", DEFAULT_SEED, "| output:", BASE)


## 1. Install dependencies
Run once in a fresh Kaggle session.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
if INSTALL_DEPENDENCIES:
    _pip("-U", "diffusers", "transformers", "accelerate", "safetensors", "sentencepiece",
         "protobuf", "pillow", "pandas", "matplotlib", "jinja2")
    try:
        from diffusers import Flux2KleinPipeline
    except ImportError:                      # older PyPI release without Flux2 Klein
        _pip("-U", "git+https://github.com/huggingface/diffusers.git")
print("Dependencies ready.")

## 2. Environment and GPU

In [ ]:
import gc, json, time, zipfile, platform, re
from datetime import datetime, timezone
import torch, numpy as np, pandas as pd
from PIL import Image, ImageOps, ImageStat
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Enable a GPU in Kaggle Settings.")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory/1024**3:.2f} GiB")
import diffusers; print("Diffusers:", diffusers.__version__)

## 3. Patient frontal prompt, generation settings and directories

In [ ]:
FRONTAL_PROMPT_DEFAULT = """
Photorealistic clinical-style frontal portrait photograph of a
completely fictional synthetic adult East Asian woman, approximately
27 years old.

PATIENT IDENTITY:
A unique new female identity, clearly different from previously
generated patients.

She has a softly heart-shaped face, gently defined cheekbones,
a delicate jawline and a small rounded chin. Light-medium beige
skin with natural skin texture, dark brown almond-shaped eyes,
naturally straight medium-thickness eyebrows and medium-full lips.

NOSE ANATOMY:
A naturally low-to-medium nasal bridge, gently curved nasal dorsum,
moderate nasal tip projection, a softly rounded nasal tip,
slightly wider alar base and naturally shaped nostrils.
The nose should be anatomically realistic and proportionate
to her facial structure.

HAIR:
Straight, dark black hair, shoulder-length, neatly tucked behind
both ears, with a natural center parting.
No glasses, jewelry or facial accessories.

CAMERA AND COMPOSITION:
Perfectly straight frontal view at 0 degrees.
Head upright, centered and facing directly toward the camera.
Both eyes looking directly at the camera.
Neutral relaxed expression, lips gently closed.
Both ears reasonably balanced in visibility.
Shoulders partially visible and level.

CLINICAL PHOTOGRAPHY:
Standardized clinical rhinoplasty preoperative photograph.
Plain light-grey background.
Soft, uniform studio lighting.
Realistic skin pores, fine facial details and natural shadows.
Sharp focus, accurate facial anatomy and photorealistic appearance.

Generate exactly one person.
No artistic stylization, no exaggerated facial features,
no extra faces, no text and no watermark.
"""

if USE_BACKEND_MANIFEST and MANIFEST_SEARCH_ROOT.exists() and list(MANIFEST_SEARCH_ROOT.glob("**/manifest.json")):
    MANIFEST_PATH = sorted(MANIFEST_SEARCH_ROOT.glob("**/manifest.json"))[0]
    m = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    assert len(m.get("prompts", [])) == 1 and len(m.get("seeds", [])) == 1, "manifest needs exactly 1 prompt and 1 seed"
    batch_raw, frontal_prompt, frontal_seed = str(m.get("batch_id", DEFAULT_BATCH_ID)), m["prompts"][0], int(m["seeds"][0])
    print("Using backend manifest:", MANIFEST_PATH)
else:
    MANIFEST_PATH = None
    batch_raw, frontal_prompt, frontal_seed = DEFAULT_BATCH_ID, FRONTAL_PROMPT_DEFAULT, int(DEFAULT_SEED)

batch_id = "".join(c if c.isalnum() or c in "-_" else "-" for c in batch_raw).strip("-") or "batch-001"
_m = re.match(r"^(P\d+)(?:[_-]|$)", batch_id, flags=re.IGNORECASE)
PATIENT_ID = _m.group(1).upper() if _m else "P01"

PATIENT_DIR = BASE / "patients" / PATIENT_ID
FRONTAL_DIR, VIEWPOINT_DIR, METADATA_DIR = PATIENT_DIR/"frontal", PATIENT_DIR/"viewpoints", PATIENT_DIR/"metadata"
BATCH_DIR = BASE / "batches" / batch_id
BATCH_IMAGE_DIR, LOG_DIR = BATCH_DIR/"images", BASE/"logs"
for d in [FRONTAL_DIR, VIEWPOINT_DIR, METADATA_DIR, BATCH_IMAGE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Batch:", batch_id, "| Patient:", PATIENT_ID, "| frontal seed:", frontal_seed, "| dir:", PATIENT_DIR)


## 4. Load FLUX.2 Klein 4B once
The same pipeline is reused for the frontal image and all eight views.

In [ ]:
from diffusers import Flux2KleinPipeline
DTYPE = torch.float16   # T4-compatible
try:
    pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, dtype=DTYPE)
except TypeError:
    pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()
print("Loaded once:", MODEL_ID)

## 5. Generate the canonical frontal image

In [ ]:
def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

frontal_path = FRONTAL_DIR / "frontal_face.png"
frontal_record = {"patient_id": PATIENT_ID, "filename": frontal_path.name, "prompt": frontal_prompt,
                  "seed": frontal_seed, "width": FRONTAL_WIDTH, "height": FRONTAL_HEIGHT,
                  "steps": FRONTAL_STEPS, "status": "FAILED", "error": None}
try:
    gen = torch.Generator(device="cpu").manual_seed(frontal_seed)
    with torch.inference_mode():
        out = pipe(prompt=frontal_prompt, height=FRONTAL_HEIGHT, width=FRONTAL_WIDTH,
                   num_inference_steps=FRONTAL_STEPS, guidance_scale=FRONTAL_GUIDANCE, generator=gen)
    frontal_image = out.images[0].convert("RGB")
    frontal_image.save(frontal_path); frontal_image.save(BATCH_IMAGE_DIR / "image_0001.png")
    frontal_record.update(status="GENERATED", path=str(frontal_path))
    display(frontal_image)
except Exception as exc:
    frontal_record["error"] = f"{type(exc).__name__}: {exc}"
    raise RuntimeError("Frontal generation failed; viewpoints will not run.") from exc
finally:
    clear_gpu_memory()

## 6. Validate and hand off the canonical frontal image
Every view is edited from this exact image (never from another generated view).

In [ ]:
if not frontal_path.is_file():
    raise FileNotFoundError(f"Frontal image missing: {frontal_path}")
canonical_reference_image = ImageOps.exif_transpose(Image.open(frontal_path)).convert("RGB")
if min(canonical_reference_image.size) < 256:
    raise ValueError(f"Frontal image too small: {canonical_reference_image.size}")
print("Canonical reference:", frontal_path, canonical_reference_image.size)
display(canonical_reference_image)

## 7. Viewpoint prompts
One separate prompt per view, built in the same 7 sections: target viewpoint → head/camera → facial perspective → identity → nasal anatomy → clinical photography → exclusions. Critical viewpoint text comes first so it can never be truncated.

In [ ]:
IDENTITY_BLOCK = """IDENTITY:
The exact same fictional adult as the supplied reference image: same age, face shape, skin tone, eyes, eyebrows, lips, jaw, chin, cheeks, hairline and hairstyle. Not a similar-looking different person. Do not beautify, stylize or reconstruct the face."""

NASAL_BLOCK = """NASAL ANATOMY:
Keep the exact nose of the reference: nasal root, bridge height and dorsum contour, length, tip shape and projection, alar width, columella, nostril shape and natural asymmetry. Do not beautify, narrow, widen, shorten, lift or reshape it; only the viewing angle changes how it appears."""

CLINICAL_BLOCK_OBLIQUE = """CLINICAL PHOTOGRAPHY:
Standardized preoperative rhinoplasty photograph, photorealistic. Eye-level camera, neutral relaxed expression, lips gently closed, eyes looking straight ahead along the head direction. Head and shoulders in frame, plain light-grey background, soft even lighting, natural skin texture, sharp focus. One person, no accessories, no text, no watermark."""

HEAD_UPRIGHT = "The head stays upright with no tilt, no chin lift or drop, and level shoulders."

OBLIQUE = {
 20: dict(
  target="VIEW: 20-DEGREE OBLIQUE CLINICAL PHOTOGRAPH, slight turn toward the {NOSE} edge of the image.",
  orient="HEAD AND CAMERA:\nThe camera stays fixed straight in front of the patient at eye level. The patient's head turns about {deg} degrees around its vertical axis so that the nose tip points toward the {NOSE} edge of the image. This is a real head rotation, not a frontal face with a sideways gaze and not a camera orbit. It is a slight but unmistakable turn from the frontal reference, far short of a three-quarter view. {upright}",
  persp="FACIAL PERSPECTIVE:\nThe nose no longer sits on the facial midline as in a frontal photograph: the tip shifts a little toward the {NOSE} edge and stays well inside the face outline. Both eyes are fully visible; the eye on the {EAR} side of the image looks slightly wider than the eye on the {NOSE} side. The cheek on the {EAR} side is slightly broader and its ear slightly more visible; the cheek on the {NOSE} side stays clearly visible.",
  excl="AVOID:\nA frontal portrait, a three-quarter view, a turn toward the {EAR} edge of the image, or head tilt."),
 40: dict(
  target="VIEW: 40-DEGREE THREE-QUARTER OBLIQUE CLINICAL PHOTOGRAPH, face turned toward the {NOSE} edge of the image.",
  orient="HEAD AND CAMERA:\nThe camera stays fixed straight in front of the patient at eye level. The patient's head turns about {deg} degrees around its vertical axis so that the nose tip points clearly toward the {NOSE} edge of the image. This is a real head rotation, not an eye movement and not a camera orbit. The turn is distinctly stronger than a 20-degree oblique and clearly short of a profile. {upright}",
  persp="FACIAL PERSPECTIVE:\nClassic three-quarter view. The nose tip approaches the cheek outline on the {NOSE} side without leaving the face. The cheek, ear and eye on the {EAR} side face the camera and look broad; the cheek on the {NOSE} side is foreshortened and its eye is visibly narrower but still fully visible. The facial midline is clearly displaced and the two sides of the face are clearly asymmetric.",
  excl="AVOID:\nA frontal portrait, a slight 20-degree turn, a full profile, a turn toward the {EAR} edge of the image, or head tilt."),
 60: dict(
  target="VIEW: 60-DEGREE LATERAL-OBLIQUE CLINICAL PHOTOGRAPH, strong turn toward the {NOSE} edge of the image, not a full profile.",
  orient="HEAD AND CAMERA:\nThe camera stays fixed straight in front of the patient at eye level. The patient's head turns strongly, about {deg} degrees around its vertical axis, so that the nose tip points emphatically toward the {NOSE} edge of the image. This is a real head rotation, not a camera orbit. {upright}",
  persp="FACIAL PERSPECTIVE:\nStrong lateral-oblique perspective approaching a profile. The nose tip reaches the cheek outline on the {NOSE} side, so the dorsum and tip projection are readable in perspective; the nose, lips and chin show clear lateral-oblique foreshortening. The cheek, ear and eye on the {EAR} side are prominent; the cheek on the {NOSE} side is strongly foreshortened and only a narrow sliver of its eye remains visible. The face is clearly narrower on that side than in a 40-degree view, yet has not become a full profile.",
  excl="AVOID:\nA frontal portrait, a 40-degree view, a full 90-degree profile, a turn toward the {EAR} edge of the image, or head tilt."),
}

def build_oblique(tier, nose_edge, deg, include_excl=True):
    ear_edge = "RIGHT" if nose_edge == "LEFT" else "LEFT"
    d = OBLIQUE[tier]
    fmt = dict(NOSE=nose_edge, EAR=ear_edge, deg=deg, upright=HEAD_UPRIGHT)
    parts = [d["target"].format(**fmt), d["orient"].format(**fmt), d["persp"].format(**fmt),
             IDENTITY_BLOCK, NASAL_BLOCK, CLINICAL_BLOCK_OBLIQUE]
    if include_excl: parts.append(d["excl"].format(**fmt))
    return "\n\n".join(parts)

BASAL = """VIEW: TRUE INFERIOR BASAL (SUBNASAL, WORM'S-EYE) CLINICAL VIEW.

HEAD AND CAMERA:
The camera is positioned BELOW the nasal tip, near the level of the upper neck, and aimed steeply upward at the underside of the nose, in a pronounced worm's-eye perspective. Camera position is the primary requirement: do not fake this view by tilting the head back in front of an eye-level camera. The nasal base faces the lens. The facial midline is vertical and centered with no left or right turn and no roll; the head may extend back naturally, without exaggerated neck distortion.

FACIAL PERSPECTIVE:
The underside of the nose is the dominant feature, and its tip is the closest point to the lens. Both nostril apertures are clearly visible from below as two distinct openings divided by the columella, framed by both alar rims, with the alar base, philtrum and upper lip below. The chin and lower face recede behind and below the nasal base in strong upward perspective. Brows, eyes and forehead appear small at the top of the frame and do not dominate.

IDENTITY:
The exact same fictional adult as the supplied reference image: same age, skin tone, hair, face shape and facial proportions. Do not beautify or reconstruct the face.

NASAL ANATOMY:
Keep the exact nose of the reference: tip shape and projection, alar width, columella, nostril shape, bridge and natural asymmetry. Do not redesign or reshape it.

CLINICAL PHOTOGRAPHY:
Standardized basal rhinoplasty photograph, photorealistic, plain light-grey background, soft even lighting, natural skin texture, sharp focus. One person, no text, no watermark.

AVOID:
An eye-level frontal portrait, a slightly upward-looking frontal portrait, a three-quarter view, a view from above, or a face where the eyes and forehead dominate. The result must unmistakably be a true inferior basal view."""

HELI = """VIEW: HIGH-ANGLE HELICOPTER (ELEVATED FRONTAL) CLINICAL VIEW.

HEAD AND CAMERA:
The camera is positioned ABOVE the forehead and in front of the face, aimed downward at about 40 degrees below the horizontal. Camera elevation is the primary requirement: the picture must show more of the upper facial surfaces than an ordinary frontal photograph. The patient gently bows the head forward, chin lowered toward the chest and forehead moving slightly toward the camera, with a natural neck and no exaggerated bend. The facial midline stays vertical and centered with no left or right turn and no roll.

FACIAL PERSPECTIVE:
The upper forehead and hairline are prominent and foreshortened in the upper part of the frame. The nasal dorsum is seen from above with the nose tip projecting toward the camera, and the tip, both alar sides and the nostrils are visible as far as this angle allows. Brows and orbits form a curved line; both cheeks, the mouth and the chin stay visible, with the chin lower in the frame. The eyelids are relaxed and both eyes look downward toward the floor, with no eye contact with the camera.

IDENTITY:
The exact same fictional adult as the supplied reference image: same age, skin tone, hair, eyes, eyebrows, face shape and proportions. Do not beautify or reconstruct the face.

NASAL ANATOMY:
Keep the exact nose of the reference: bridge, dorsum contour, tip shape and projection, alar width, nostril shape and natural asymmetry. Do not redesign or reshape it.

CLINICAL PHOTOGRAPHY:
Standardized rhinoplasty photograph, photorealistic, neutral relaxed expression, lips gently closed, head and shoulders in frame, plain light-grey background, soft even lighting, natural skin texture, sharp focus. One person, no text, no watermark.

AVOID:
An eye-level frontal portrait, a worm's-eye view, a side profile, or an extreme vertical bird's-eye view. The elevated camera and downward gaze must be unmistakable."""


# ---- one constant per view -------------------------------------------------------
LEFT_20_PROMPT  = build_oblique(20, "LEFT",  PROMPT_DEGREES["left_20"])
LEFT_40_PROMPT  = build_oblique(40, "LEFT",  PROMPT_DEGREES["left_40"])
LEFT_60_PROMPT  = build_oblique(60, "LEFT",  PROMPT_DEGREES["left_60"])
RIGHT_20_PROMPT = build_oblique(20, "RIGHT", PROMPT_DEGREES["right_20"])
RIGHT_40_PROMPT = build_oblique(40, "RIGHT", PROMPT_DEGREES["right_40"])
RIGHT_60_PROMPT = build_oblique(60, "RIGHT", PROMPT_DEGREES["right_60"])
BASAL_PROMPT, HELICOPTER_PROMPT = BASAL, HELI

# Order and numbering identical to the end-to-end notebook (keeps seeds reproducible)
VIEWPOINT_PROMPT_STRATEGIES = {
    "left_20":    {"name": "Left 20°",   "nose_toward": "image-left",  "prompt": LEFT_20_PROMPT},
    "left_40":    {"name": "Left 40°",   "nose_toward": "image-left",  "prompt": LEFT_40_PROMPT},
    "left_60":    {"name": "Left 60°",   "nose_toward": "image-left",  "prompt": LEFT_60_PROMPT},
    "right_20":   {"name": "Right 20°",  "nose_toward": "image-right", "prompt": RIGHT_20_PROMPT},
    "right_40":   {"name": "Right 40°",  "nose_toward": "image-right", "prompt": RIGHT_40_PROMPT},
    "right_60":   {"name": "Right 60°",  "nose_toward": "image-right", "prompt": RIGHT_60_PROMPT},
    "basal":      {"name": "Basal",      "nose_toward": "n/a (camera below nose)", "prompt": BASAL_PROMPT},
    "helicopter": {"name": "Helicopter", "nose_toward": "n/a (camera above head)", "prompt": HELICOPTER_PROMPT},
}
VIEWS = list(VIEWPOINT_PROMPT_STRATEGIES)
print("Configured views:", VIEWS)

### Prompt audit (runs before any view is generated)
Checks sections, left/right consistency and the 512-token limit using the pipeline's own tokenizer.

In [ ]:
def count_prompt_tokens(prompt):
    tk = getattr(pipe, "tokenizer", None)
    text = tk.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                  add_generation_prompt=True, enable_thinking=False)
    return len(tk(text)["input_ids"])

SECTIONS = ["VIEW:", "HEAD AND CAMERA:", "FACIAL PERSPECTIVE:", "IDENTITY:",
            "NASAL ANATOMY:", "CLINICAL PHOTOGRAPHY:", "AVOID:"]
rows = []
for key, s in VIEWPOINT_PROMPT_STRATEGIES.items():
    p = s["prompt"]
    missing = [h for h in SECTIONS if h not in p]
    assert not missing, f"{key}: missing sections {missing}"
    if key.startswith(("left", "right")):
        want = "LEFT" if key.startswith("left") else "RIGHT"
        nose = re.findall(r"nose tip points (?:clearly |emphatically )?toward the (LEFT|RIGHT) edge", p)
        assert nose == [want], f"{key}: nose direction {nose}, expected {want}"
        assert re.search(r"anatomical|patient's own|viewer's", p) is None, f"{key}: anatomical/ambiguous direction wording"
    n = count_prompt_tokens(p)
    assert n <= MAX_PROMPT_TOKENS, f"{key}: {n} tokens > {MAX_PROMPT_TOKENS} - end of prompt would be truncated"
    rows.append({"view": key, "nose_toward": s["nose_toward"], "prompt_degrees": PROMPT_DEGREES.get(key, "-"), "tokens": n})
# left/right prompts must be exact mirrors of each other (no blind copy errors)
for t in (20, 40, 60):
    l, r = VIEWPOINT_PROMPT_STRATEGIES[f"left_{t}"]["prompt"], VIEWPOINT_PROMPT_STRATEGIES[f"right_{t}"]["prompt"]
    l = re.sub(r"about \d+ degrees", "about N degrees", l); r = re.sub(r"about \d+ degrees", "about N degrees", r)
    assert l.replace("LEFT", "@").replace("RIGHT", "LEFT").replace("@", "RIGHT") == r, f"{t}: left/right not mirrored"
display(pd.DataFrame(rows))
print("Prompt audit passed.")

## 8. Image validation and view generation engine

In [ ]:
def validate_generated_image(path, reference, min_resolution=(256, 256)):
    # Integrity checks only - they cannot judge viewpoint, identity or anatomy.
    res = {"is_valid": False, "reason": ""}
    if not path.is_file(): res["reason"] = "Output file does not exist."; return res
    try: img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    except Exception as e: res["reason"] = f"Unreadable image: {e}"; return res
    if img.width < min_resolution[0] or img.height < min_resolution[1]:
        res["reason"] = f"Resolution too small: {img.size}"; return res
    if float(np.mean(ImageStat.Stat(img).stddev)) < 5:
        res["reason"] = "Image appears blank."; return res
    ref = reference.resize(img.size)
    if float(np.mean(np.abs(np.asarray(img, np.float32) - np.asarray(ref, np.float32)))) < 2:
        res["reason"] = "Output almost identical to frontal reference."; return res
    res.update(is_valid=True, reason="Integrity checks passed; manual viewpoint review required.")
    return res

def view_seed(view_key, attempt, rerun=False):
    base = int(frontal_seed) + VIEW_SEED_OFFSET + (RERUN_SEED_OFFSET if rerun else 0)
    return base + VIEWS.index(view_key) * VIEW_SEED_STRIDE + attempt * ATTEMPT_SEED_STRIDE

def generate_view(view_key, reference=None, rerun=False, force=False):
    reference = reference if reference is not None else canonical_reference_image
    s = VIEWPOINT_PROMPT_STRATEGIES[view_key]
    final_path = VIEWPOINT_DIR / f"{PATIENT_ID}_{view_key}.png"
    rec = {"patient_id": PATIENT_ID, "view_key": view_key, "view_name": s["name"],
           "nose_toward": s["nose_toward"], "prompt_degrees": PROMPT_DEGREES.get(view_key),
           "filename": final_path.name, "filepath": str(final_path), "status": "FAILED",
           "prompt_used": s["prompt"], "attempt_number": 0, "seed": None, "steps": VIEW_STEPS,
           "error": None, "validation_reason": None, "attempt_history": []}
    if SKIP_EXISTING and not force and view_key not in FORCE_REGENERATE_VIEWS and final_path.exists():
        v = validate_generated_image(final_path, reference)
        if v["is_valid"]:
            rec.update(status="EXISTING_REQUIRES_REVIEW", validation_reason=v["reason"]); return rec
    for n in range(1, max(1, min(int(MAX_PROMPT_ATTEMPTS), 3)) + 1):
        seed = view_seed(view_key, n, rerun)
        temp = VIEWPOINT_DIR / f".{PATIENT_ID}_{view_key}_attempt_{n}.png"
        att = {"attempt": n, "seed": seed, "status": "FAILED", "error": None}
        print(f"[{view_key}] attempt {n} | seed {seed} | steps {VIEW_STEPS}")
        clear_gpu_memory()
        try:
            gen = torch.Generator(device="cpu").manual_seed(seed)
            with torch.inference_mode():
                out = pipe(image=reference, prompt=s["prompt"], height=VIEW_HEIGHT, width=VIEW_WIDTH,
                           num_inference_steps=VIEW_STEPS, guidance_scale=VIEW_GUIDANCE,
                           max_sequence_length=512, generator=gen)
            out.images[0].convert("RGB").save(temp, format="PNG")
            v = validate_generated_image(temp, reference, (min(256, VIEW_WIDTH), min(256, VIEW_HEIGHT)))
            att["validation_reason"] = v["reason"]
            if v["is_valid"]:
                os.replace(temp, final_path); att["status"] = "GENERATED_REQUIRES_REVIEW"
                rec.update(status="GENERATED_REQUIRES_REVIEW", attempt_number=n, seed=seed, error=None,
                           validation_reason=v["reason"])
                rec["attempt_history"].append(att); return rec
            att["error"] = "Validation failed: " + v["reason"]
        except torch.cuda.OutOfMemoryError as e:
            att["error"] = f"CUDA OOM: {e}"; rec["error"] = "Stopped after CUDA OOM."; rec["attempt_history"].append(att)
            temp.unlink(missing_ok=True); clear_gpu_memory(); return rec
        except Exception as e:
            att["error"] = f"{type(e).__name__}: {e}"; print("Generation error:", att["error"])
        finally:
            temp.unlink(missing_ok=True); clear_gpu_memory()
        rec["attempt_history"].append(att)
    rec["error"] = rec["error"] or "All attempts failed."
    return rec

## 9. Generate all eight views
To add a future view: add its prompt constant and one entry in `VIEWPOINT_PROMPT_STRATEGIES`.

In [ ]:
viewpoint_execution_results = []
for view_key in VIEWS:
    print(f"\n--- {VIEWPOINT_PROMPT_STRATEGIES[view_key]['name']} ---")
    viewpoint_execution_results.append(generate_view(view_key))
print("\nView generation completed.")

## 10. Visual review (all views together)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
panels = [("FRONTAL reference", canonical_reference_image)]
for k in VIEWS:
    p = VIEWPOINT_DIR / f"{PATIENT_ID}_{k}.png"
    panels.append((f"{VIEWPOINT_PROMPT_STRATEGIES[k]['name']}  (nose → {VIEWPOINT_PROMPT_STRATEGIES[k]['nose_toward']})",
                   Image.open(p).convert("RGB") if p.exists() else None))
for ax, (title, im) in zip(axes.ravel(), panels):
    ax.axis("off"); ax.set_title(title, fontsize=11)
    if im is not None: ax.imshow(im)
plt.tight_layout(); plt.show()
print("Manually check: nose direction (left views -> image-left, right views -> image-right), 20/40/60 clearly "
      "different, identity and nose anatomy kept, basal = camera below nose, helicopter = camera above.")

## 11. Failed views and targeted rerun

In [ ]:
failed = [r["view_key"] for r in viewpoint_execution_results if r["status"] == "FAILED"]
print("Failed views:", failed)

def rerun_views(view_keys=None, force=True):
    """Regenerate only the given views (default: failed ones) with fresh seeds from the same frontal image."""
    keys = view_keys or [r["view_key"] for r in viewpoint_execution_results if r["status"] == "FAILED"]
    for k in keys:
        new = generate_view(k, rerun=True, force=force)
        for i, old in enumerate(viewpoint_execution_results):
            if old["view_key"] == k: viewpoint_execution_results[i] = new
    return viewpoint_execution_results
# Example: rerun_views(["right_60"])   # after editing PROMPT_DEGREES and re-running the prompt cell

## 12. Manifest, metadata, generation log and ZIP

In [ ]:
rows = [{"patient_id": PATIENT_ID, "viewtype": "frontal", "view_name": "Frontal Baseline", "filename": frontal_path.name,
         "filepath": str(frontal_path), "status": frontal_record["status"], "seed": frontal_seed,
         "steps": FRONTAL_STEPS, "prompt_degrees": None, "nose_toward": None, "attempt_number": 1,
         "prompt_used": frontal_prompt, "requires_manual_review": True,
         "validation_reason": "Frontal baseline; manual identity/anatomy review required.", "error": frontal_record["error"]}]
for r in viewpoint_execution_results:
    ok = Path(r["filepath"]).is_file()
    rows.append({"patient_id": PATIENT_ID, "viewtype": r["view_key"], "view_name": r["view_name"], "filename": r["filename"],
                 "filepath": r["filepath"] if ok else "", "status": r["status"] if ok else "MISSING_FILE", "seed": r["seed"],
                 "steps": r["steps"], "prompt_degrees": r["prompt_degrees"], "nose_toward": r["nose_toward"],
                 "attempt_number": r["attempt_number"], "prompt_used": r["prompt_used"], "requires_manual_review": True,
                 "validation_reason": r["validation_reason"], "error": r["error"]})
manifest_df = pd.DataFrame(rows)
manifest_csv_path = METADATA_DIR / "manifest.csv"
manifest_df.to_csv(manifest_csv_path, index=False, encoding="utf-8")

metadata_doc = {"batch_id": batch_id, "patient_id": PATIENT_ID, "model_id": MODEL_ID,
                "created_at_utc": datetime.now(timezone.utc).isoformat(), "frontal_seed": frontal_seed,
                "settings": {"view_size": [VIEW_WIDTH, VIEW_HEIGHT], "view_steps": VIEW_STEPS, "view_guidance": VIEW_GUIDANCE,
                             "prompt_degrees": PROMPT_DEGREES},
                "frontal_generation": frontal_record, "viewpoint_generation": viewpoint_execution_results,
                "note": "Integrity checks do not verify pose, identity or nasal anatomy. Manual review is mandatory."}
metadata_path = METADATA_DIR / "metadata.json"
metadata_path.write_text(json.dumps(metadata_doc, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
generation_log_path = LOG_DIR / f"{PATIENT_ID}_generation_log.json"
generation_log_path.write_text(json.dumps(metadata_doc, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
(BATCH_DIR / "metadata.json").write_text(json.dumps(metadata_doc, indent=2, default=str), encoding="utf-8")

zip_path = BASE / f"{PATIENT_ID}_synthetic_face_dataset.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in PATIENT_DIR.rglob("*"):
        if f.is_file(): zf.write(f, arcname=str(f.relative_to(BASE)))
display(manifest_df[["patient_id", "viewtype", "view_name", "nose_toward", "seed", "status", "filename"]])
print("ZIP:", zip_path, f"({zip_path.stat().st_size/1024**2:.1f} MB)")
# Write batch_summary.json for backend compatibility
batch_summary_doc = {
    "job_id": int(m.get("job_id", 0)) if MANIFEST_PATH else 0,
    "batch_id": batch_id,
    "patient_id": PATIENT_ID,
    "requested_count": len(rows),
    "successful_count": sum(1 for r in rows if r["status"] in ("GENERATED", "GENERATED_REQUIRES_REVIEW", "EXISTING_REQUIRES_REVIEW")),
    "failed_count": sum(1 for r in rows if r["status"] == "FAILED"),
    "images": [
        {
            "filename": r["filename"],
            "seed": r["seed"],
            "prompt": r["prompt_used"],
            "viewtype": r["viewtype"],
            "view_name": r["view_name"],
            "status": r["status"],
            "path": r["filepath"],
            "error": r["error"]
        }
        for r in rows
    ]
}
(BATCH_DIR / "batch_summary.json").write_text(json.dumps(batch_summary_doc, indent=2, default=str), encoding="utf-8")
(BASE / "batch_summary.json").write_text(json.dumps(batch_summary_doc, indent=2, default=str), encoding="utf-8")


## 13. Final summary and GPU cleanup

In [ ]:
print("Batch:", batch_id, "| Patient:", PATIENT_ID)
print("Frontal:", frontal_path.exists(),
      "| Views:", sum((VIEWPOINT_DIR / f"{PATIENT_ID}_{v}.png").exists() for v in VIEWS), "/", len(VIEWS))
print("Failed views:", [r["view_key"] for r in viewpoint_execution_results if r["status"] == "FAILED"])
print("Archive:", zip_path)
print("All images require manual review - prompts cannot guarantee exact angles, identity or anatomy.")
# To free the GPU when completely finished (rerun_views() needs the pipeline, so do this last):
# del pipe; clear_gpu_memory()